# Chapter 1 &mdash; Problem, Procedure, Algorithm

**Concept 2 of the Chapter 1 decomposition:** *Problem vs. Procedure vs. Algorithm*

A procedure is anything mechanisable. An algorithm is a procedure that <b>halts on every input</b>. Fermat's Last Theorem lets us watch the same problem have only a procedure, then an algorithm.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Problem-Procedure-Algorithm/Concept-Problem-Procedure-Algorithm.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


* A **problem** can be stated without knowing how to solve it.
* A **procedure** is anything mechanisable &mdash; it may run forever.
* An **algorithm** is a procedure that **halts on all of its expected inputs**.

Fermat asked for naturals $a,b,c > 1$ with $a^n + b^n = c^n$ for $n > 2$.
Before 1995 we had a *procedure*: search systematically, forever.
After Wiles' proof we have an *algorithm*: print "impossible" and halt.

**The problem never changed. Only halting did.**

## 2. Definitions

### The procedure: search for a Fermat triple

Enumerate $a,b,c$ by increasing sum, testing each. Perfectly mechanical &mdash; and on
$n>2$ it never stops, because Wiles proved there is nothing to find.

We give it a `budget` so the notebook returns. **The budget is our impatience, not
part of the procedure.**

In [ ]:
def fermat_search(n, budget=200000):
    """Procedure: look for a,b,c > 1 with a^n + b^n = c^n.
       Returns the triple, or None if the budget ran out.
       For n > 2 it would run forever if we let it."""
    steps = 0
    total = 6                       # smallest possible a+b+c with all > 1
    while True:
        for a in range(2, total):
            for b in range(2, total - a):
                c = total - a - b
                if c < 2:
                    continue
                steps += 1
                if steps > budget:
                    return None     # out of budget -- NOT a proof of anything
                if a**n + b**n == c**n:
                    return (a, b, c)
        total += 1

### The algorithm (post-1995)

Because the answer is known, we can decide in constant time. This halts on **every**
input, so it is an algorithm.

In [ ]:
def fermat_decide(n):
    """Algorithm: always halts, always correct (thanks to Wiles 1995)."""
    if n <= 2:
        return "solutions exist"
    return "impossible to find such a,b,c"

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;1.&nbsp;The Motivating Question: Can Computers Do Everything?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Can-Computers-Do-Everything/Concept-Can-Computers-Do-Everything.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;3.&nbsp;Hilbert's Program, and its Refutation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Hilbert-Undecidable-Incomplete/Concept-Hilbert-Undecidable-Incomplete.ipynb)&nbsp;&rarr;

---

## 3. Tests

For $n = 2$ the procedure **succeeds and halts** &mdash; Pythagorean triples exist.

In [ ]:
print("n=2 :", fermat_search(2))       # finds a triple, e.g. (3,4,5) up to ordering
assert fermat_search(2) is not None, "Pythagorean triples exist"

For $n = 3$ it exhausts the budget and returns `None`.

Read that result carefully: **`None` does not mean "no solution exists".** It means
*we stopped looking*. That gap between "did not find" and "does not exist" is the
entire subject of Chapters 14 and 15.

In [ ]:
print("n=3 :", fermat_search(3, budget=50000), "   <- budget exhausted, NOT a proof")
print("n=4 :", fermat_search(4, budget=50000))
assert fermat_search(3, budget=50000) is None   # not a proof -- just no luck

The algorithm, by contrast, answers instantly for every input we throw at it.

In [ ]:
for n in [1, 2, 3, 4, 17, 1000]:
    print("n =", n, "->", fermat_decide(n))
assert all(fermat_decide(n) for n in range(1, 50))   # ALWAYS halts

## 4. Exercises


1. Raise the budget on `fermat_search(3, ...)`. Does any budget let you conclude
   there is no solution? Explain why not, in one sentence.
2. `fermat_search` enumerates by increasing $a+b+c$. Why does it matter that the
   enumeration reaches **every** triple eventually? (Compare Chapter 3's
   *numeric order*.)
3. Write a procedure that halts exactly when a given number is *not* prime.
   Is it an algorithm? Now write one that halts exactly when it *is* prime.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter1/Concept-Problem-Procedure-Algorithm')